In [3]:
"""
Alert Management System - Gmail-Compatible HTML Emails
"""

import os
import smtplib
import requests
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.image import MIMEImage
from datetime import datetime
from dotenv import load_dotenv
from geopy.geocoders import Nominatim
import time

load_dotenv()


class AlertManager:
    def __init__(self, db_manager=None):
        self.db = db_manager
        self.email_enabled = os.getenv('ENABLE_EMAIL', 'false').lower() == 'true'
        self.smtp_server = os.getenv('SMTP_SERVER', 'smtp.gmail.com')
        self.smtp_port = int(os.getenv('SMTP_PORT', 587))
        self.email_from = os.getenv('EMAIL_FROM')
        self.email_password = os.getenv('EMAIL_PASSWORD')
        self.alert_email_to = os.getenv('ALERT_EMAIL_TO')
        self.phone_ip = os.getenv('PHONE_IP', '192.168.0.105')
        self.camera_lat = float(os.getenv('CAMERA_LATITUDE', '12.90111'))
        self.camera_lon = float(os.getenv('CAMERA_LONGITUDE', '77.51806'))
        self.camera_location = os.getenv('CAMERA_LOCATION', 'RNS Institute, Bangalore')
        self.geolocator = Nominatim(user_agent="waste_detection_v1")
        self.alert_threshold = 1
        self.email_sent_today = 0
        self.max_emails_per_day = 50
        self.location_cache = None
        self.cache_expiry = 300
        self.last_location_update = 0
        
        print(f"Alert Manager: Email {'Enabled' if self.email_enabled else 'Disabled'}")
    
    def get_ip_based_location(self):
        try:
            r = requests.get('http://ip-api.com/json/', timeout=5)
            if r.status_code == 200:
                d = r.json()
                if d.get('status') == 'success':
                    return {
                        'latitude': d.get('lat', self.camera_lat),
                        'longitude': d.get('lon', self.camera_lon),
                        'area': d.get('city', 'Unknown'),
                        'city': d.get('city', 'Unknown'),
                        'region': d.get('regionName', ''),
                        'country': d.get('country', ''),
                        'source': 'ip_geolocation'
                    }
        except:
            pass
        return None
    
    def get_phone_gps(self):
        return None
    
    def get_address_from_coords(self, lat, lon):
        try:
            loc = self.geolocator.reverse(f"{lat}, {lon}", timeout=10)
            if loc: return loc.address
        except:
            pass
        return self.camera_location
    
    def get_current_location_info(self):
        t = time.time()
        if self.location_cache and (t - self.last_location_update) < self.cache_expiry:
            return self.location_cache
        
        loc = self.get_ip_based_location()
        if not loc:
            loc = {
                'latitude': self.camera_lat,
                'longitude': self.camera_lon,
                'area': 'Channasandra',
                'city': 'Bangalore',
                'region': 'Karnataka',
                'country': 'India',
                'source': 'fixed'
            }
        
        self.location_cache = loc
        self.last_location_update = t
        return loc
    
    def should_send_alert(self, count):
        return self.email_enabled and count >= self.alert_threshold and self.email_sent_today < self.max_emails_per_day
    
    def send_email_alert(self, detection_id, waste_count, confidence, image_path=None):
        if not self.email_enabled:
            return False
        
        try:
            loc = self.get_current_location_info()
            
            # Colors
            if waste_count >= 10:
                col = "#d32f2f"
                bg = "#ffebee"
                priority = "HIGH"
            elif waste_count >= 5:
                col = "#ff9800"
                bg = "#fff3e0"
                priority = "MEDIUM"
            else:
                col = "#4caf50"
                bg = "#e8f5e9"
                priority = "LOW"
            
            # Create message
            msg = MIMEMultipart('alternative')
            msg['Subject'] = f"🚨 {waste_count} Plastic Item{'s' if waste_count>1 else ''} - {loc.get('area', 'Detection')}"
            msg['From'] = self.email_from
            msg['To'] = self.alert_email_to
            
            # Text version
            text = f"""WASTE ALERT - {priority} PRIORITY

Detection #{detection_id}
Items: {waste_count}
Confidence: {confidence:.1%}
Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S IST')}

Location: {loc.get('area')}, {loc.get('city')}
GPS: {loc['latitude']:.6f}, {loc['longitude']:.6f}

Map: https://www.google.com/maps?q={loc['latitude']},{loc['longitude']}

RNS Institute of Technology - AI/ML Department"""
            
            # HTML version (simplified for Gmail)
            html = f"""<html>
<body style="margin:0;padding:20px;font-family:Arial,sans-serif;background:#f5f5f5;">
<div style="max-width:600px;margin:0 auto;background:white;border-radius:10px;overflow:hidden;box-shadow:0 2px 4px rgba(0,0,0,0.1);">

<div style="background:{col};color:white;padding:30px;text-align:center;">
<h1 style="margin:0;">🚨 Waste Detection Alert</h1>
<p style="margin:10px 0 0;">AI-Powered Monitoring System</p>
</div>

<div style="background:{bg};border-left:5px solid {col};padding:20px;margin:20px;">
<h2 style="margin:0;color:{col};">🔴 {priority} PRIORITY</h2>
<p style="margin:10px 0 0;color:#555;">Plastic waste detected - Action required</p>
</div>

<div style="padding:20px;">
<h3 style="color:#1976d2;border-bottom:2px solid #1976d2;padding-bottom:8px;">📊 Detection Details</h3>
<table style="width:100%;font-size:14px;" cellpadding="8">
<tr><td style="font-weight:bold;width:40%;">Detection ID:</td><td>#{detection_id}</td></tr>
<tr><td style="font-weight:bold;">Plastic Items:</td><td><strong style="background:#fff9c4;padding:2px 8px;border-radius:3px;">{waste_count} item{'s' if waste_count>1 else ''}</strong></td></tr>
<tr><td style="font-weight:bold;">Confidence:</td><td>{confidence:.1%}</td></tr>
<tr><td style="font-weight:bold;">Time:</td><td>{datetime.now().strftime('%B %d, %Y %I:%M %p IST')}</td></tr>
</table>
</div>

<div style="padding:20px;background:#fafafa;">
<h3 style="color:#1976d2;border-bottom:2px solid #1976d2;padding-bottom:8px;">📍 Location</h3>
<table style="width:100%;font-size:14px;" cellpadding="8">
<tr><td style="font-weight:bold;width:40%;">Area:</td><td><strong>{loc.get('area', 'Unknown')}</strong></td></tr>
<tr><td style="font-weight:bold;">City:</td><td>{loc.get('city', 'Unknown')}</td></tr>
<tr><td style="font-weight:bold;">GPS:</td><td>{loc['latitude']:.6f}°N, {loc['longitude']:.6f}°E</td></tr>
<tr><td style="font-weight:bold;">Map:</td><td><a href="https://www.google.com/maps?q={loc['latitude']},{loc['longitude']}" style="background:#4caf50;color:white;padding:8px 16px;text-decoration:none;border-radius:5px;display:inline-block;">View on Maps</a></td></tr>
</table>
</div>

<div style="background:#263238;color:#b0bec5;padding:20px;text-align:center;font-size:12px;">
<strong style="color:white;">AI-Powered Waste Detection System</strong><br>
RNS Institute of Technology - AI/ML Department<br>
Alert #{self.email_sent_today + 1} of {self.max_emails_per_day}<br>
Contact: wastedetection.project@gmail.com
</div>

</div>
</body>
</html>"""
            
            # Attach parts
            msg.attach(MIMEText(text, 'plain'))
            msg.attach(MIMEText(html, 'html'))
            
            # Attach image
            if image_path and os.path.exists(image_path):
                with open(image_path, 'rb') as f:
                    img = MIMEImage(f.read())
                    img.add_header('Content-Disposition', 'attachment', filename=os.path.basename(image_path))
                    msg.attach(img)
            
            # Send
            with smtplib.SMTP(self.smtp_server, self.smtp_port) as s:
                s.starttls()
                s.login(self.email_from, self.email_password.strip())
                s.send_message(msg)
            
            self.email_sent_today += 1
            print(f"✅ Email sent ({self.email_sent_today}/{self.max_emails_per_day})")
            return True
            
        except Exception as e:
            print(f"❌ Email error: {e}")
            return False
    
    def trigger_alert(self, detection_id, waste_count, confidence, image_path=None):
        if self.should_send_alert(waste_count):
            return self.send_email_alert(detection_id, waste_count, confidence, image_path)
        return False
    
    def reset_daily_counter(self):
        self.email_sent_today = 0
